# Day 10: MHA：拆头与合头

线性投影改变内容；view 拆分 H×D；transpose 交换 head/time 轴。各 head 输出合并后经过 output projection 回到残差宽度。E 不必等于 H×D。

参考源码：`learning/mini_transformer/model.py`。

每日概念、误区、面试问答：[知识讲义](../docs/KNOWLEDGE_DAY08_14.md)。

当前个人进度以[进度记录](../docs/LEARNING_PROGRESS.md)为准；运行参考答案不等于已独立掌握。

[Day 10完整讲义](../docs/DAY10_MULTIHEAD.md)：概念、计算契约、源码实验、调试优化、答案与资料。

先写自己的预测，再执行原有代码单元格；不要自动勾选个人验收。

## 1. 图解与数字例子

把每个头理解成同一段资料上的不同查询方式，不是把一句话分给几个人各看一截。

```text
输入每位置E=8个数
   ↓ Q投影：8 → Hq*D=2*6=12
每位置12个特征 → 2个头，每头6个
   ↓ 各自计算
2张T_q×T_kv匹配表
   ↓ 各头汇集V
合并为12维 → 输出投影回8维 → 接回残差
```

若B=1、T=3，score是 `[1,2,3,3]`，共18个数，
不是把两个头串成一张6×6的跨头大表。
Hq*D不必等于E；**先Linear计算新特征，后view重新分组**。

## 先预测

执行前写下预期 shape、输出或调度顺序；在下面另建 Markdown 单元格保存自己的答案，不修改参考实现作为唯一证据。

In [ ]:
import json
import os
import sys
from pathlib import Path
root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/curriculum.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import torch
from learning.curriculum import run_day
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


In [ ]:
report = run_day(10, DEVICE)
print(json.dumps(report, indent=2, ensure_ascii=False))
destination = Path('logs') / f"day{report['day']:02d}-course.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n')
print('Report:', destination)


## 自测与失败实验

四个 head 的 score 是四张 T×T 表，还是一张 4T×4T？

记录你改变的唯一变量、预期、实际结果和差异原因。

`requires_cuda` / `requires_cuda_compiler` / `external_service_not_configured` 都不是通过。

## 5. 自测与参考答案

**问：2个头、每头6维，E=8合法吗？**
答：合法，只要投影为12维并在输出时投影回8维；D=E/Hq只适用于额外规定投影同宽的设计。

**问：为什么是两张3×3表？**
答：每个头在自己的投影空间匹配位置，不让头0的Query直接参加头1的softmax。

**问：输出投影只是降维吗？**
答：还有混合不同头特征的作用，即使输入输出同宽也能保留。

**问：transpose是否等于拷贝所有数据？**
答：一般是视图与stride变化，后续操作是否复制需查实际布局与算子。

完整实现限制、调试步骤与资料见[本日讲义](../docs/DAY10_MULTIHEAD.md)。

以上是参考答案，不是个人学习结果。

## 验收记录

- [ ] 能解释原理
- [ ] 能自行复现
- [ ] 能解释一次失败
- [ ] 能说明当前实现不支持什么

个人笔记：

待填写。